In [1]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

## Connect to postgres
Use your own credentials

In [2]:
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

cursor = conn.cursor()
print("Connection established!")

Connection established!


## Load dataset

In [3]:
df = pd.read_csv(os.getenv("DATA_PATH"), encoding="latin-1")
print(f"Dataset loaded: {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

Dataset loaded: 180519 rows and 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


### DIMENSION: dim_product

In [4]:
# Select the relevant columns and drop duplicates to create the dim_product DataFrame
dim_product = df[["Product Card Id", "Product Name", "Category Name", "Department Name", "Product Price"]].drop_duplicates()

# Rename columns to match the dim_product table schema
dim_product.columns = ["product_id", "product_name", "category", "department", "unit_cost"]

# Insert the data into the dim_product table
execute_values(cursor, """
    INSERT INTO dim_product (product_id, product_name, category, department, unit_cost)
    VALUES %s
    ON CONFLICT (product_id) DO NOTHING
""", dim_product.values.tolist())

conn.commit()
print(f"dim_product populated: {len(dim_product)} products inserted")

dim_product populated: 118 products inserted


### DIMENSION: dim_customer

In [5]:
# Select relevant columns and remove duplicates
dim_customer = df[["Customer Id", "Customer Segment", "Market", "Customer Country"]].drop_duplicates()

# Rename columns to match SQL table
dim_customer.columns = ["customer_id", "customer_segment", "market", "country"]

# Insert into PostgreSQL
execute_values(cursor, """
    INSERT INTO dim_customer (customer_id, customer_segment, market, country)
    VALUES %s
    ON CONFLICT (customer_id) DO NOTHING
""", dim_customer.values.tolist())

conn.commit()
print(f"dim_customer populated: {len(dim_customer)} customers inserted")

dim_customer populated: 43816 customers inserted


### DIMENSION: dim_region

In [6]:
# Select relevant columns and remove duplicates
dim_region = df[["Order Region", "Order Country", "Market"]].drop_duplicates().reset_index(drop=True)

# Add a region_id manually
dim_region.insert(0, "region_id", range(1, len(dim_region) + 1))

# Rename columns to match SQL table
dim_region.columns = ["region_id", "region", "country", "market"]

# Insert into PostgreSQL
execute_values(cursor, """
    INSERT INTO dim_region (region_id, region, country, market)
    VALUES %s
    ON CONFLICT (region_id) DO NOTHING
""", dim_region.values.tolist())

conn.commit()
print(f"dim_region populated: {len(dim_region)} regions inserted")

dim_region populated: 167 regions inserted


### DIMENSION: dim_date


In [8]:
# Convert to datetime first, then strip time component
df["order date (DateOrders)"] = pd.to_datetime(df["order date (DateOrders)"])
df["order_date_clean"] = df["order date (DateOrders)"].dt.date

# Extract unique dates and build the dimension
dim_date = pd.DataFrame(df["order_date_clean"].drop_duplicates()).reset_index(drop=True)
dim_date.insert(0, "date_id", range(1, len(dim_date) + 1))
dim_date.columns = ["date_id", "full_date"]

# Convert back to datetime for attribute extraction
temp = pd.to_datetime(dim_date["full_date"])
dim_date["year"]       = temp.dt.year
dim_date["quarter"]    = temp.dt.quarter
dim_date["month"]      = temp.dt.month
dim_date["month_name"] = temp.dt.strftime("%B")
dim_date["week"]       = temp.dt.isocalendar().week.astype(int)
dim_date["is_weekend"] = temp.dt.dayofweek >= 5

# Insert into PostgreSQL
execute_values(cursor, """
    INSERT INTO dim_date (date_id, full_date, year, quarter, month, month_name, week, is_weekend)
    VALUES %s
    ON CONFLICT (date_id) DO NOTHING
""", dim_date.values.tolist())

conn.commit()
print(f"dim_date populated: {len(dim_date)} dates inserted")

dim_date populated: 1127 dates inserted


### FACT TABLE: fact_orders

In [9]:
# Add clean date column to main dataframe
df["order_date_clean"] = pd.to_datetime(df["order date (DateOrders)"]).dt.date

# Get date_id lookup
date_lookup = dim_date.set_index("full_date")["date_id"]

# Fix region lookup — use region + country as key
region_lookup = dim_region.set_index(["region", "country"])["region_id"]

# Build fact_orders
fact_orders = df[[
    "Order Id",
    "Customer Id",
    "Product Card Id",
    "order_date_clean",
    "Order Region",
    "Order Country",
    "Order Item Quantity",
    "Order Item Product Price",
    "Sales",
    "Order Item Profit Ratio"
]].copy()

fact_orders.columns = [
    "order_id", "customer_id", "product_id",
    "full_date", "region", "country", "quantity_ordered",
    "unit_price", "total_revenue", "profit_margin"
]

# Map dimension IDs
fact_orders["date_id"]   = fact_orders["full_date"].map(date_lookup)
fact_orders["region_id"] = fact_orders.set_index(["region", "country"]).index.map(region_lookup)

# Select final columns in correct order
fact_orders = fact_orders[[
    "order_id", "customer_id", "product_id",
    "date_id", "region_id", "quantity_ordered",
    "unit_price", "total_revenue", "profit_margin"
]].drop_duplicates(subset=["order_id"])

# Insert into PostgreSQL
execute_values(cursor, """
    INSERT INTO fact_orders (order_id, customer_id, product_id, date_id, region_id,
                             quantity_ordered, unit_price, total_revenue, profit_margin)
    VALUES %s
    ON CONFLICT (order_id) DO NOTHING
""", fact_orders.values.tolist())

conn.commit()
print(f"fact_orders populated: {len(fact_orders)} orders inserted")

fact_orders populated: 65752 orders inserted


### FACT TABLE: fact_shipments

In [12]:
conn.rollback()
print("Connection reset")

Connection reset


In [13]:
# Recreate clean date column
df["order_date_clean"] = pd.to_datetime(df["order date (DateOrders)"]).dt.date

# Recreate date_lookup from PostgreSQL
cursor.execute("SELECT date_id, full_date FROM dim_date")
date_lookup = {row[1]: row[0] for row in cursor.fetchall()}

# Build fact_shipments
fact_shipments = df[[
    "Order Id",
    "order_date_clean",
    "Shipping Mode",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Late_delivery_risk",
    "Order Item Total"
]].copy()

fact_shipments.columns = [
    "order_id", "full_date", "shipping_mode",
    "days_for_shipping_real", "days_for_shipment_scheduled",
    "late_delivery_flag", "shipping_cost"
]

# Convert 0/1 to boolean
fact_shipments["late_delivery_flag"] = fact_shipments["late_delivery_flag"].astype(bool)

# Map date_id
fact_shipments["date_id"] = fact_shipments["full_date"].map(date_lookup)

# Add shipment_id
fact_shipments = fact_shipments.drop_duplicates(subset=["order_id"]).reset_index(drop=True)
fact_shipments.insert(0, "shipment_id", range(1, len(fact_shipments) + 1))

# Select final columns in correct order
fact_shipments = fact_shipments[[
    "shipment_id", "order_id", "date_id", "shipping_mode",
    "days_for_shipping_real", "days_for_shipment_scheduled",
    "late_delivery_flag", "shipping_cost"
]]

# Insert into PostgreSQL
execute_values(cursor, """
    INSERT INTO fact_shipments (shipment_id, order_id, date_id, shipping_mode,
                                days_for_shipping_real, days_for_shipment_scheduled,
                                late_delivery_flag, shipping_cost)
    VALUES %s
    ON CONFLICT (shipment_id) DO NOTHING
""", fact_shipments.values.tolist())

conn.commit()
print(f"fact_shipments populated: {len(fact_shipments)} shipments inserted")

fact_shipments populated: 65752 shipments inserted
